# 🤖 AI Engineering Fundamentals — Lezione 6
## Notebook Gruppo B

**ITS Novitas 4.0 | Martedì 09/06/2026 🏁**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic, os, json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente

def chiedi_claude(messaggio, system=None, max_tokens=600, temperature=0.3):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

SYSTEM_WIDATA = """
Sei l'assistente di WiData Srl, startup IoT di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì 'Non ho questa informazione.'
"""

DOCUMENTO_WIDATA = """
SENSORE XS200: temperatura -20°C/+60°C, umidità 0-100%, IP67, batteria 2 anni, LoRaWAN/WiFi.
GATEWAY GW500: gestisce 1000 sensori, copertura 15km rurale/3km urbano, storage 32GB.
XPLORE: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).
SUPPORTO: lun-ven 9-18, support@widata.cloud, +39 079 123456.
"""

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Valutazione — AI-as-a-Judge

Costruite un sistema di valutazione automatica per il chatbot WiData.
Usate Claude per valutare le risposte di Claude — AI-as-a-Judge.

---
### Esercizio 1 — AI-as-a-Judge base *(guidato)*

Implementate la funzione `valuta_risposta()` che usa Claude
per valutare le risposte del chatbot su 5 criteri.

In [ ]:
# Esercizio 1 — valuta_risposta()

def valuta_risposta(domanda, risposta, contesto=""):
    """AI-as-a-Judge: Claude valuta la risposta del chatbot."""

    prompt = f"""
Sei un valutatore esperto di sistemi AI. Valuta questa risposta di un chatbot.

DOMANDA: {domanda}
CONTESTO RAG: {contesto[:400] if contesto else 'Nessuno'}
RISPOSTA: {risposta}

Valuta su questi criteri (1-5):
- pertinenza: quanto è rilevante alla domanda?
- accuratezza: è basata sul contesto e corretta?
- completezza: risponde in modo esauriente?
- hallucination: inventa informazioni? (5=no hallucination, 1=molte)
- tono: è professionale e appropriato?

Rispondi SOLO con JSON valido, nessun testo aggiuntivo:
{{"pertinenza": X, "accuratezza": X, "completezza": X,
  "hallucination": X, "tono": X, "nota": "commento breve"}}
"""
    # Il giudice deve essere deterministico → temperature=0.0
    risultato_testo = chiedi_claude(prompt, temperature=0.0)

    try:
        return json.loads(risultato_testo)
    except json.JSONDecodeError:
        # Fallback: cerca il JSON nel testo
        import re
        match = re.search(r'\{.*\}', risultato_testo, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"errore": "JSON non parsato", "raw": risultato_testo[:200]}

def stampa_valutazione(domanda, valutazione):
    """Stampa la valutazione in modo leggibile."""
    if "errore" in valutazione:
        print(f"  ❌ Errore: {valutazione['errore']}")
        return
    criteri = ["pertinenza", "accuratezza", "completezza", "hallucination", "tono"]
    media = sum(valutazione.get(c, 0) for c in criteri) / len(criteri)
    print(f"  Pertinenza:    {valutazione.get('pertinenza', '?')}/5")
    print(f"  Accuratezza:   {valutazione.get('accuratezza', '?')}/5")
    print(f"  Completezza:   {valutazione.get('completezza', '?')}/5")
    print(f"  Hallucination: {valutazione.get('hallucination', '?')}/5")
    print(f"  Tono:          {valutazione.get('tono', '?')}/5")
    print(f"  Media:         {media:.1f}/5")
    print(f"  Nota:          {valutazione.get('nota', '')}")

# Test con 3 casi
casi_test = [
    {
        "domanda": "Qual è l'autonomia del sensore XS200?",
        "risposta": "Il sensore XS200 ha un'autonomia di 2 anni a batteria.",
        "contesto": DOCUMENTO_WIDATA
    },
    {
        "domanda": "Quanto costa il piano Pro di Xplore?",
        "risposta": "Il piano Pro costa €49 al mese e supporta fino a 100 sensori.",
        "contesto": DOCUMENTO_WIDATA
    },
    {
        "domanda": "Offrite sensori subacquei?",
        "risposta": "Certo! Abbiamo una gamma completa di sensori subacquei fino a 500m.",
        "contesto": DOCUMENTO_WIDATA  # hallucination — non è nel documento!
    },
]

for caso in casi_test:
    print(f"\n{'='*55}")
    print(f"❓ {caso['domanda']}")
    print(f"🤖 {caso['risposta']}")
    print()
    val = valuta_risposta(caso["domanda"], caso["risposta"], caso["contesto"])
    stampa_valutazione(caso["domanda"], val)

# Nota: il terzo caso (sensori subacquei) dovrebbe ricevere un punteggio di
# hallucination basso (≈1-2), perché inventa informazioni non presenti nel documento.

---
### Esercizio 2 — Dataset di valutazione *(guidato)*

Create un dataset di 10 Q&A per il chatbot WiData.
Valutate tutte le risposte e calcolate la media per ogni metrica.

In [ ]:
# Esercizio 2 — dataset di valutazione completo

def genera_risposta_chatbot(domanda):
    """Genera una risposta dal chatbot con RAG simulato."""
    prompt = f"Contesto:\n{DOCUMENTO_WIDATA}\n\nDomanda: {domanda}"
    return chiedi_claude(prompt, system=SYSTEM_WIDATA)

dataset = [
    "Qual è l'autonomia della batteria del sensore XS200?",
    "Quanti sensori gestisce il gateway GW500?",
    "Quanto costa il piano Pro di Xplore?",
    "Il sensore XS200 è impermeabile?",
    "Come si contatta il supporto tecnico?",
    "Qual è la copertura del gateway in area urbana?",
    "Offrite un piano gratuito?",
    "WiData ha sensori per ambienti subacquei?",      # non nel doc
    "Qual è il prezzo del piano Enterprise?",           # non specificato
    "Dove si trova la sede di WiData?",                # non nel doc
]

print("📊 Valutazione dataset completo\n")
print("=" * 70)

risultati = []
criteri = ["pertinenza", "accuratezza", "completezza", "hallucination", "tono"]

for i, domanda in enumerate(dataset):
    print(f"\nDomanda {i+1}/{len(dataset)}: {domanda[:50]}...")

    # 1. Il chatbot genera la risposta (con contesto RAG)
    risposta = genera_risposta_chatbot(domanda)

    # 2. L'AI-as-a-Judge valuta la risposta
    val = valuta_risposta(domanda, risposta, DOCUMENTO_WIDATA)

    if "errore" not in val:
        media = sum(val.get(c, 0) for c in criteri) / len(criteri)
        print(f"  Media: {media:.1f}/5 | Hallucination: {val.get('hallucination', '?')}/5")
        risultati.append(val)

# Report finale
if risultati:
    print(f"\n{'='*70}")
    print("📈 REPORT FINALE")
    print(f"{'='*70}")
    for criterio in criteri:
        media_c = sum(r.get(criterio, 0) for r in risultati) / len(risultati)
        print(f"  {criterio:<15}: {media_c:.2f}/5")
    media_tot = sum(sum(r.get(c, 0) for c in criteri)/len(criteri) for r in risultati) / len(risultati)
    print(f"\n  Score totale: {media_tot:.2f}/5")

    print("\n💡 Quale criterio ha il punteggio più basso?")
    print("   Cosa modifichereste per migliorarlo?")
    # Tipicamente i criteri più bassi sono accuratezza/hallucination sulle domande
    # NON presenti nel documento (sede, prezzo Enterprise, subacquei): il chatbot
    # dovrebbe rispondere "Non ho questa informazione". Per migliorare: rafforzare
    # l'istruzione anti-hallucination e/o aggiungere quei dati al documento.

---
### Esercizio 3 — Bias dell'AI-as-a-Judge *(libero)*

Huyen Cap. 4 descrive 4 bias noti del giudice AI.
Dimostrate sperimentalmente almeno 2:

- **Verbosity bias**: il giudice preferisce risposte più lunghe?
- **Self-preference**: Claude preferisce le proprie risposte?
- **Position bias**: preferisce la prima risposta in un confronto?

Per ogni bias: costruite un esperimento, raccogliete dati, traete conclusioni.

In [ ]:
# Esercizio 3 — dimostrare i bias

# ── BIAS 1: Verbosity bias ──────────────────────────────────────────
# Stessa risposta in versione breve e lunga — il giudice valuta meglio quella lunga?

domanda = "Qual è l'autonomia del sensore XS200?"
contesto = DOCUMENTO_WIDATA
criteri = ["pertinenza", "accuratezza", "completezza", "hallucination", "tono"]

risposta_breve = "2 anni."
risposta_lunga = (
    "Il sensore XS200 di WiData Srl è dotato di una batteria Li-Ion da 3.7V "
    "che garantisce un'autonomia operativa di ben 2 anni senza necessità di "
    "manutenzione o sostituzione. Questo lo rende particolarmente adatto per "
    "installazioni in luoghi difficilmente accessibili dove la manutenzione "
    "frequente sarebbe problematica e costosa."
)

def media_valutazione(domanda, risposta, contesto):
    val = valuta_risposta(domanda, risposta, contesto)
    if "errore" in val:
        return None, val
    return sum(val.get(c, 0) for c in criteri) / len(criteri), val

print("TEST VERBOSITY BIAS")
media_breve, _ = media_valutazione(domanda, risposta_breve, contesto)
print(f"Risposta breve ({len(risposta_breve)} char): media {media_breve}/5")

media_lunga, _ = media_valutazione(domanda, risposta_lunga, contesto)
print(f"Risposta lunga ({len(risposta_lunga)} char): media {media_lunga}/5")

print("\nConclusione: entrambe le risposte contengono la stessa informazione corretta")
print("(2 anni). Se la risposta lunga ottiene una media più alta della breve, il giudice")
print("mostra 'verbosity bias': tende a premiare la lunghezza/ricchezza di dettagli")
print("anche quando il contenuto fattuale è identico (es. su completezza e tono).")

In [ ]:
# ── BIAS 2: Position bias ───────────────────────────────────────────
# Stesse due risposte in ordine A poi B, poi in ordine B poi A
# Il giudice preferisce sempre la prima?

def valuta_comparative(domanda, risposta_a, risposta_b, contesto=""):
    """Chiede al giudice quale risposta è migliore."""
    prompt = f"""
Domanda: {domanda}
Contesto: {contesto[:300]}

Risposta 1: {risposta_a}
Risposta 2: {risposta_b}

Quale risposta è migliore? Rispondi SOLO con '1' o '2'.
"""
    return chiedi_claude(prompt, temperature=0.0).strip()

r1 = "Il sensore XS200 dura 2 anni a batteria."
r2 = "L'autonomia è di circa 24 mesi grazie alla batteria Li-Ion."

print("TEST POSITION BIAS — 5 esperimenti")
ordine_normale = 0
ordine_invertito = 0

for i in range(5):
    # Ordine normale: r1 prima
    scelta_normale = valuta_comparative(domanda, r1, r2, DOCUMENTO_WIDATA)
    # Ordine invertito: r2 prima
    scelta_invertita = valuta_comparative(domanda, r2, r1, DOCUMENTO_WIDATA)

    print(f"  Esperimento {i+1}: ordine normale → {scelta_normale} | invertito → {scelta_invertita}")

print("\nIl giudice tende a preferire sempre la risposta 1 indipendentemente dal contenuto?")
# ...

---
### Esercizio 4 — Guardrail automatico *(libero)*

Costruite un guardrail che usa AI-as-a-Judge per verificare
le risposte prima di mostrarle all'utente.
Se il punteggio di hallucination è < 3, blocca la risposta
e mostra un messaggio di errore.

In [ ]:
# Esercizio 4 — guardrail con AI-as-a-Judge

SOGLIA_HALLUCINATION = 3  # blocca se hallucination < 3

def chat_con_guardrail(domanda, contesto=DOCUMENTO_WIDATA):
    """Chatbot con guardrail automatico anti-hallucination."""

    # 1. Genera la risposta
    prompt = f"Contesto:\n{contesto}\n\nDomanda: {domanda}"
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)

    # 2. Valutiamo la risposta con l'AI-as-a-Judge
    valutazione = valuta_risposta(domanda, risposta, contesto)

    # 3. Se il punteggio di hallucination è sotto la soglia, blocchiamo la risposta
    if valutazione.get("hallucination", 5) < SOGLIA_HALLUCINATION:
        return f"⚠️ RISPOSTA BLOCCATA (hallucination score: {valutazione.get('hallucination')}/5)\n"\
               f"Motivo: {valutazione.get('nota', 'risposta non affidabile')}"

    # 4. Risposta approvata
    score = valutazione.get('hallucination', '?')
    return f"{risposta}\n\n[✅ Guardrail: hallucination {score}/5]"

# Test
domande_test = [
    "Qual è l'autonomia del sensore XS200?",     # risposta corretta — passa
    "Offrite sensori GPS integrati?",              # probabile hallucination — bloccata?
    "Qual è il fatturato di WiData nel 2025?",    # inventata — bloccata?
]

for d in domande_test:
    print(f"\n{'='*55}")
    print(f"❓ {d}")
    print(chat_con_guardrail(d))

# Conclusione:
# Il guardrail blocca le risposte con basso punteggio di hallucination prima di
# mostrarle all'utente. Per le domande fuori documento, se il chatbot risponde
# correttamente "Non ho questa informazione" il giudice darà hallucination alta
# (nessuna invenzione) e la risposta passerà. Possibili falsi positivi: risposte
# corrette ma giudicate male; per questo conviene tarare la SOGLIA e tenere il
# giudice a temperature=0 per renderlo più stabile.

---
## 📊 Preparate la presentazione (5 slide)

1. **Cos'è AI-as-a-Judge** — perché valutare un LLM è diverso da ML classico
2. **I risultati del dataset** — tabella con score per ogni criterio
3. **I bias dimostrati** — i vostri esperimenti con i numeri
4. **Il guardrail** — come funziona e i risultati sui test
5. **La vostra conclusione** — quando fidarsi dell'AI-as-a-Judge?

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*